In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

from tri_ct_tools.image.checker import plot_intensity
from tri_ct_tools.image.reader import singlecam_series, singlecam_mean
from tri_ct_tools.convert.holdup import two_point_holdup, plot_holdup

%matplotlib widget

In [ ]:
# plot with gas fraction calculated through mean intensities
def holdup_of_mean(meas_path, full_path, empty_path, frames, img_shape=(1548, 1524)):
    Imeas = singlecam_mean(meas_path, frames, img_shape)
    Ifull = singlecam_mean(full_path, frames, img_shape)
    Iempty = singlecam_mean(empty_path, frames, img_shape)
    holdup = two_point_holdup(Imeas, Ifull, Iempty)
    return holdup
    # plot_intensity(holdup, threshold=0.5, vmin=0, vmax=0.2, title="Holdup from mean")


In [ ]:
# plot with gas fraction calculated on instantaneous inensities, then averaged
def mean_of_holdup(meas_path, full_path, empty_path, frames, img_shape=(1548, 1524)):
    Imeas = singlecam_series(meas_path, frames, img_shape)
    Ifull = singlecam_mean(full_path, frames, img_shape)
    Iempty = singlecam_mean(empty_path, frames, img_shape)
    holdup = two_point_holdup(Imeas, Ifull, Iempty)
    mean_holdup = np.mean(holdup, 0)
    return mean_holdup

In [ ]:
meas_folder= Path(R"U:\Xray RPT ChemE\X-ray\Xray_data\2025-06-27 Rik\02_preprocessed\1500x1500Crop_150lmin_150kV_22Hz\camera 2")
full_folder = Path(R"U:\Xray RPT ChemE\X-ray\Xray_data\2025-06-27 Rik\02_preprocessed\1500x1500Crop_Full_150kV_22Hz\camera 2")
empty_folder = Path(R"U:\Xray RPT ChemE\X-ray\Xray_data\2025-06-27 Rik\02_preprocessed\1500x1500Crop_Empty_150kV_22Hz\camera 2")

frames = range(50, 1200)

img_shape = (1524, 1548)

Ifull = singlecam_mean(full_folder, frames, img_shape)
Iempty = singlecam_mean(empty_folder, frames, img_shape)
Imeas_mean = singlecam_mean(meas_folder, frames, img_shape)
Imeas_series = singlecam_series(meas_folder, frames, img_shape)


In [ ]:
hu_mean = two_point_holdup(Imeas_mean, Ifull, Iempty)

mean_hu = np.mean(two_point_holdup(Imeas_series, Ifull, Iempty), axis=0)


In [ ]:
plot_holdup(hu_mean, title="Holdup from mean", vmax=0.15)
plot_holdup(mean_hu, title="Mean of holdup", vmax=0.15)

In [ ]:
hu_diff = hu_mean - mean_hu
fig, im= plot_holdup(hu_diff, title="absolute difference (hu from mean - mean of hu)", vmin=-0.005, vmax=0.005, colormap=mpl.colormaps["PuOr"])
fig.colorbar(im)

In [ ]:
hu_diff_norm = hu_diff / np.mean(hu_mean) * 100
fig, im = plot_holdup(hu_diff_norm, title="Normalized difference (%)", vmin=-10, vmax=10, colormap=mpl.colormaps['PRGn'])
fig.colorbar(im)